# Fashion MNIST Image Classification using CNN

**Author:** Your Name  
**Course:** Deep Learning / Computer Vision  
**Assignment:** No. 3

---

## Project Overview

In this project, we build a **Convolutional Neural Network (CNN)** using TensorFlow/Keras to classify grayscale images of clothing items from the **Fashion MNIST** dataset into 10 categories.

### Goals
- Load and preprocess the Fashion MNIST dataset
- Design and train a CNN model
- Evaluate performance using accuracy, confusion matrix, and classification report
- Visualize training curves and sample predictions

### Dataset
Fashion MNIST contains **70,000 images** (28×28 grayscale), split into:
- 60,000 training samples
- 10,000 testing samples
- 10 clothing categories: T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle Boot

## Setup

Install required libraries if not already available.

In [ ]:
%pip install matplotlib scikit-learn tensorflow --quiet

## Task 1 — Data Loading & Preprocessing

We load Fashion MNIST directly from `tf.keras.datasets`, then:
- **Normalize** pixel values from [0, 255] to [0, 1] — helps the model converge faster
- **Reshape** images to (28, 28, 1) to add the channel dimension required by Conv2D layers

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Load dataset
fashion_mnist = tf.keras.datasets.fashion_mnist
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

print('Training Images Shape:', X_train.shape)
print('Training Labels Shape:', y_train.shape)
print('Testing Images Shape :', X_test.shape)
print('Testing Labels Shape :', y_test.shape)

# Normalize pixel values to [0, 1]
X_train = X_train / 255.0
X_test  = X_test  / 255.0

# Reshape to (samples, 28, 28, 1) — Conv2D expects a channel dimension
X_train = X_train.reshape(-1, 28, 28, 1)
X_test  = X_test.reshape(-1, 28, 28, 1)

# Class labels
class_names = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle Boot'
]

print('\nAfter Preprocessing:')
print('Training Shape:', X_train.shape)
print('Testing Shape :', X_test.shape)

### Sample Images

Let's visualize a few training images to understand what the dataset looks like.

In [ ]:
plt.figure(figsize=(10, 5))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(X_train[i].reshape(28, 28), cmap='gray')
    plt.title(class_names[y_train[i]], fontsize=11)
    plt.axis('off')
plt.suptitle('Sample Training Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Task 2 — CNN Model Architecture

We use a Sequential CNN with the following design decisions:

| Layer | Purpose |
|---|---|
| `Conv2D(32)` | Detect low-level features (edges, textures) |
| `MaxPooling2D` | Reduce spatial dimensions, retain important features |
| `Conv2D(64)` | Detect higher-level patterns |
| `MaxPooling2D` | Further downsampling |
| `Dropout(0.25)` | Prevent overfitting after conv layers |
| `Flatten` | Convert 2D feature maps to 1D vector |
| `Dense(128)` | Learn complex combinations of features |
| `Dropout(0.5)` | Stronger regularization before final layer |
| `Dense(10, softmax)` | Output probability for each of 10 classes |

> **Why Dropout?** Dropout randomly deactivates neurons during training, forcing the network to learn redundant representations and reducing overfitting.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([
    # Block 1: First conv + pooling
    Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D((2, 2)),

    # Block 2: Deeper conv + pooling
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Dropout(0.25),  # Regularization

    # Classifier head
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),   # Stronger regularization before output
    Dense(10, activation='softmax')  # 10 output classes
])

model.summary()

## Task 3 — Compiling & Training

**Optimizer:** Adam — adaptive learning rate, works well out of the box  
**Loss:** Sparse Categorical Crossentropy — used when labels are integers (not one-hot)  
**Early Stopping:** Monitors validation loss and stops training if it doesn't improve for 3 epochs — prevents overfitting and saves time  
**Training:** 20 epochs max, batch size 64, 20% validation split

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Early stopping: stop if val_loss doesn't improve for 3 consecutive epochs
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,  # Revert to the best model weights
    verbose=1
)

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.20,
    callbacks=[early_stop]
)

### Training Curves

Plotting accuracy and loss across epochs helps us see:
- Whether the model is learning (accuracy should rise, loss should fall)
- Whether it's overfitting (training accuracy >> validation accuracy)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'],     label='Training Accuracy',   linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2, linestyle='--')
axes[0].set_title('Accuracy over Epochs', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'],     label='Training Loss',   linewidth=2, color='orange')
axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2, color='red', linestyle='--')
axes[1].set_title('Loss over Epochs', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Training vs Validation Performance', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## Task 4 — Model Evaluation

We evaluate on the **held-out test set** (never seen during training) to get an unbiased measure of real-world performance.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Loss:     {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy*100:.2f}%')

In [ ]:
# Generate predictions on test set
predictions = model.predict(X_test)
y_pred = np.argmax(predictions, axis=1)  # Convert probabilities to class indices

### Confusion Matrix

A confusion matrix shows which classes are being predicted correctly and which are being confused with others. Bright diagonal = good performance.

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.ticker as ticker

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(range(10), class_names, rotation=45, ha='right', fontsize=9)
plt.yticks(range(10), class_names, fontsize=9)

# Annotate each cell with the count
for i in range(10):
    for j in range(10):
        plt.text(j, i, str(cm[i, j]),
                 ha='center', va='center',
                 color='white' if cm[i, j] > cm.max() / 2 else 'black',
                 fontsize=8)

plt.tight_layout()
plt.show()

### Classification Report

Per-class **precision**, **recall**, and **F1-score** give a much more detailed picture than overall accuracy alone — especially useful for identifying which classes the model struggles with.

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(y_test, y_pred, target_names=class_names)
print('Classification Report:')
print('=' * 60)
print(report)

## Task 5 — Sample Predictions

Let's visually inspect how the model performs on individual test images.  
**Green title** = correct prediction, **Red title** = incorrect prediction.

In [ ]:
plt.figure(figsize=(14, 8))

for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_test[i].reshape(28, 28), cmap='gray')

    pred  = class_names[y_pred[i]]
    true  = class_names[y_test[i]]
    color = 'green' if y_pred[i] == y_test[i] else 'red'

    plt.title(f'Pred: {pred}\nTrue: {true}', color=color, fontsize=8)
    plt.axis('off')

plt.suptitle('Sample Predictions (Green = Correct, Red = Wrong)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Save the Model

We save the trained model so it can be loaded and used later without retraining.

In [ ]:
model.save('fashion_mnist_cnn.h5')
print('Model saved as fashion_mnist_cnn.h5')

# To load later:
# from tensorflow.keras.models import load_model
# model = load_model('fashion_mnist_cnn.h5')

## Summary

| Step | Details |
|---|---|
| Dataset | Fashion MNIST — 70,000 images, 10 classes |
| Preprocessing | Normalized to [0,1], reshaped to (28,28,1) |
| Architecture | 2× Conv2D + MaxPool + Dropout + Dense |
| Training | Adam optimizer, Early Stopping, 20% validation split |
| Test Accuracy | ~91–93% (varies by run) |
| Saved Model | `fashion_mnist_cnn.h5` |

### Key Takeaways
- CNNs are highly effective for image classification tasks
- Dropout is essential for reducing overfitting
- Early stopping prevents wasted training time and keeps the best model
- Confusion matrix reveals that **Shirt** and **T-shirt/top** are the most commonly confused classes (visually similar)